### Performance of Drivers

In [0]:
%sql
CREATE OR REPLACE TABLE f1_catalog.gold.driver_performance AS
SELECT 
    d.driver_id,
    d.full_name AS driver_name,

    COUNT(r.race_id) AS total_races,

    SUM(CASE WHEN r.position = 1 THEN 1 ELSE 0 END) AS total_wins,

    SUM(CASE WHEN r.position <= 3 THEN 1 ELSE 0 END) AS total_podiums,

    SUM(r.points) AS total_points,

    ROUND(SUM(r.points) / COUNT(r.race_id), 2) AS points_per_race,

    ROUND(100 * SUM(CASE WHEN r.position = 1 THEN 1 ELSE 0 END) 
          / COUNT(r.race_id), 2) AS win_percentage,

    MIN(r.position) AS best_finish,

    STDDEV(r.position) AS consistency_score

FROM f1_catalog.silver.results r
JOIN f1_catalog.silver.drivers d 
ON r.driver_id=d.driver_id

GROUP BY d.driver_id, d.full_name;

SELECT * FROM f1_catalog.gold.driver_performance LIMIT 10;

How many races they done and from them how many qualify


In [0]:
%sql
CREATE OR REPLACE TABLE f1_catalog.gold.qualifying_vs_race AS
SELECT 
    q.race_id,
    q.driver_id,
    d.full_name AS driver_name,

    q.position AS qualifying_position,
    r.position AS race_position,

    (q.position - r.position) AS position_gain

FROM f1_catalog.silver.qualifying q

LEFT JOIN f1_catalog.silver.results r
ON q.race_id = r.race_id 
AND q.driver_id = r.driver_id

JOIN f1_catalog.silver.drivers d
ON q.driver_id = d.driver_id

WHERE r.position IS NOT NULL;
select * from f1_catalog.gold.qualifying_vs_race;

Constructur standing

In [0]:
%sql
CREATE OR REPLACE TABLE f1_catalog.gold.constructor_standings AS
SELECT 
    c.constructor_id,
    c.name AS constructor_name,

    COUNT(r.race_id) AS total_races,

    SUM(CASE WHEN r.position = 1 THEN 1 ELSE 0 END) AS total_wins,

    SUM(r.points) AS total_points,

    ROUND(SUM(r.points)/COUNT(r.race_id),2) AS avg_points_per_race,

    ROUND(100 * SUM(CASE WHEN r.position = 1 THEN 1 ELSE 0 END) 
          / COUNT(r.race_id),2) AS win_ratio

FROM f1_catalog.silver.results r
JOIN f1_catalog.silver.constructors c 
ON r.constructor_id = c.constructor_id

GROUP BY c.constructor_id, c.name;

select * from f1_catalog.gold.constructor_standings

In [0]:
%sql
select * from f1_catalog.gold.constructor_standings
where total_wins>1;

### Summary of races

In [0]:
%sql
CREATE OR REPLACE TABLE f1_catalog.gold.race_summary AS
SELECT ra.race_id,ra.name AS race_name,ra.race_year,d.full_name AS winner,r.fastest_lap_time
FROM f1_catalog.silver.results r
JOIN f1_catalog.silver.races ra 
ON r.race_id=ra.race_id
JOIN f1_catalog.silver.drivers d 
ON r.driver_id=d.driver_id

WHERE r.position=1 and r.fastest_lap_time IS NOT NULL;

select * from f1_catalog.gold.race_summary

### Driver ranking by year

In [0]:
%sql
CREATE OR REPLACE TABLE f1_catalog.gold.driver_ranking_by_year AS
SELECT race_year,driver_name,total_points,
DENSE_RANK() OVER (PARTITION BY race_year ORDER BY total_points DESC) AS driver_rank
FROM (SELECT ra.race_year,d.full_name AS driver_name,SUM(r.points) AS total_points
FROM f1_catalog.silver.results r
JOIN f1_catalog.silver.races ra ON r.race_id=ra.race_id
JOIN f1_catalog.silver.drivers d ON r.driver_id=d.driver_id
GROUP BY ra.race_year, d.full_name
);

select * from f1_catalog.gold.driver_ranking_by_year

### Champions per year

In [0]:
%sql
CREATE OR REPLACE TABLE f1_catalog.gold.championship_winners AS
SELECT race_year,driver_name,constructor_name,total_points
FROM (SELECT ra.race_year,d.full_name AS driver_name,c.name AS constructor_name,
        SUM(r.points) AS total_points,
        DENSE_RANK() OVER (PARTITION BY ra.race_year ORDER BY SUM(r.points) DESC) AS rank
        FROM f1_catalog.silver.results r
        JOIN f1_catalog.silver.races ra
        ON r.race_id=ra.race_id
    JOIN f1_catalog.silver.drivers d
        ON r.driver_id=d.driver_id

    JOIN f1_catalog.silver.constructors c
        ON r.constructor_id=c.constructor_id

    GROUP BY ra.race_year, d.full_name, c.name
) t
WHERE rank = 1;
select * from f1_catalog.gold.championship_winners

### Circuits summary

In [0]:
%sql
CREATE OR REPLACE TABLE f1_catalog.gold.circuit_analysis AS
SELECT c.circuit_id,c.name AS circuit_name,d.driver_id,d.full_name AS driver_name,
COUNT(r.race_id) AS total_races,SUM(CASE WHEN r.position=1 THEN 1 ELSE 0 END) AS total_wins,
ROUND(AVG(r.position), 2) AS avg_position
FROM f1_catalog.silver.results r
JOIN f1_catalog.silver.races ra ON r.race_id=ra.race_id
JOIN f1_catalog.silver.circuits c ON ra.circuit_id=c.circuit_id
JOIN f1_catalog.silver.drivers d ON r.driver_id=d.driver_id
GROUP BY c.circuit_id,c.name,d.driver_id, d.full_name;

select * from f1_catalog.gold.circuit_analysis

### Performance Summary

In [0]:
%sql
CREATE OR REPLACE TABLE f1_catalog.gold.performance_score AS
SELECT driver_id,driver_name,total_points,win_percentage,points_per_race,consistency_score,
ROUND((0.4 * win_percentage) +(0.3 * points_per_race) +(0.3 * (1 / NULLIF(consistency_score, 0))), 2) AS performance_score
FROM f1_catalog.gold.driver_performance;

select * from f1_catalog.gold.performance_score

Visualization

In [0]:
%sql
SELECT driver_name,COUNT(*) AS total_championships
FROM f1_catalog.gold.championship_winners
GROUP BY driver_name
ORDER BY total_championships DESC
LIMIT 10;

Line chart of Constructor wins by decade

In [0]:
%sql
SELECT (race_year DIV 10) * 10 AS decade,constructor_name,COUNT(*) AS total_wins
FROM f1_catalog.gold.championship_winners
GROUP BY decade, constructor_name
ORDER BY decade, total_wins DESC;

In [0]:
%sql
SELECT nationality,COUNT(*) AS total_drivers
FROM f1_catalog.silver.drivers
GROUP BY nationality
ORDER BY total_drivers DESC;